In [3]:
from google import genai
import os
import chromadb
from dotenv import load_dotenv, find_dotenv
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")
client=genai.Client(api_key=api_key)

In [ ]:
from pypdf import PdfReader
reader=PdfReader("CFBP Udaap.pdf")
document_text=""
for page in reader.pages:
    document_text+=page.extract_text()
print(document_text[:1000])

In [5]:
chat=client.chats.create(model="gemini-2.5-flash",
                         config={"system_instruction":f"""You are a senior compliance officer with 10 years 
                                of experience in banking regulations, credit risk policy, and internal controls.
                                
                                You have been given the following policy document to answer questions from:
                                
                                ========================
                                {document_text}
                                ========================
                                
                                Rules:
                                - Only answer from the document above
                                - Never make up information not in the document
                                - If the answer is not in the document, say "This information is not available in the provided document"
                                - Always respond in this format:
                                
                                Policy Area: [area]
                                Answer: [your answer]
                                Source: [quote the relevant section from the document]
                                Confidence: [High/Medium/Low]
                                """,
                                "temperature": 0.2,
                                "max_output_tokens": 1024})

In [ ]:
while True:
    user_input=input("User : ")
    if user_input.lower() in ("end", "quit", "bye", "goodbye"):
        break
    response=chat.send_message(user_input)
    print( f"Model : {response.text}\n")    

In [ ]:
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

In [6]:
def chunk_text(text, chunk_size=800, overlap=100):
    """
    Splits text into overlapping chunks.
    chunk_size: max characters per chunk
    overlap: characters shared between consecutive chunks (keeps context from being cut mid-sentence)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # move forward, but overlap with previous chunk
    return chunks

In [ ]:
chunks = chunk_text(document_text)
print(f"Number of chunks: {len(chunks)}")
print("--- First chunk ---")
print(chunks[0])
print("--- Second chunk ---")
print(chunks[1])

In [ ]:
print(chromadb.__version__)
print(os.getcwd())

1.5.9
